In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Portátil\Desktop\Work\Simulacio_empr\Sprint4\Origin_data\02_Datos procesados (2)\01_Demografia_2016_GEN.csv",sep=";")
df.head()

,a01: anio de la entrevista,a01: anio de la entrevista.1,a02: mes de la entrevista,"a0000: es requisito que pregunte su genero, es vd. hombre o mujer?",a0100: en que pais nacio?,a0400: en que anio nacio?,a04: edad calculada,a0800: me podria decir su edad aproximada?,a0900a: con quien vive habitualmente?a. solo/a,a0900b: con quien vive habitualmente?b.con su pareja,...,a1300: cual es el contenido del titulo de formacion profesional que ha obtenido?,"a1400: cuantos libros habia, donde vivia cuando tenia 10 anios?",a1500: cual es su situacion laboral actual?,a1600: cual es el nivel de estudios que esta cursando?,a1700: ha trabajado alguna vez...?,a1900: su trabajo se relaciona/relacionaba con temas financieros?,a05: edad calculada de la persona informada,a0300: en que pais naciste?,a1800: profesion o trabajo (2 digitos cno-11),a0200: en que provincia nacio?
0,2016,2016,12,0,1,1998,18,-98,0,0,...,-98,2,9,6,0,-98,-98,-98,-98,1
1,2016,2016,10,0,1,1982,33,-98,0,1,...,6,3,21,-98,-98,1,-98,-98,45,1
2,2016,2016,10,1,1,1967,49,-98,0,1,...,-98,1,7,-98,1,0,-98,-98,63,1
3,2016,2016,10,0,1,1962,54,-98,0,1,...,-98,1,4,-98,1,0,-98,-98,95,1
4,2016,2016,10,0,1,1952,64,-98,0,1,...,-98,1,6,-98,1,0,-98,-98,77,1


In [2]:
#Eliminar text deixar nomes codi
df.columns = df.columns.str.split(':').str[0]
df.head(2)

,a01,a01,a02,a0000,a0100,a0400,a04,a0800,a0900a,a0900b,...,a1300,a1400,a1500,a1600,a1700,a1900,a05,a0300,a1800,a0200
0,2016,2016,12,0,1,1998,18,-98,0,0,...,-98,2,9,6,0,-98,-98,-98,-98,1
1,2016,2016,10,0,1,1982,33,-98,0,1,...,6,3,21,-98,-98,1,-98,-98,45,1


In [3]:
df.columns.tolist()

['a01',
 'a01',
 'a02',
 'a0000',
 'a0100',
 'a0400',
 'a04',
 'a0800',
 'a0900a',
 'a0900b',
 'a0900c',
 'a0900d',
 'a0900e',
 'a0900f',
 'a0900g',
 'a0900h',
 'a0900i',
 'a0900j',
 'a0900k',
 'a1000',
 'a1100',
 'a1200',
 'a1300',
 'a1400',
 'a1500',
 'a1600',
 'a1700',
 'a1900',
 'a05',
 'a0300',
 'a1800',
 'a0200']

#### Asignación al grupo por situacion laboral

In [4]:
import numpy as np

df['labor_status'] = np.where(df['a1500'] == 1, "autónomo",
                       np.where(df['a1500'] == 1, "asalariado",
                       np.where(df['a1500'] == 5, "desempleado",
                       np.where(df['a1500'] == 6, "jubilado",
                       np.where(df['a1500'].isin([4,7,8,9,10]), "otros inactivos","missing")))))

#### Asignación al grupo por estructura familiar

In [5]:
df['familia'] = np.where(
    df['a0900b'] == 1, "Vive con pareja",
    np.where(
        df['a0900a'] == 1, "Vive solo/a",
        np.where(
            (df['a0900b'] == 0) &
            (
                (df['a0900d'] == 1) |
                (df['a0900e'] == 1) |
                (df['a0900j'] == 1) |
                (df['a0900h'] == 1) |
                (df['a0900g'] == 1) |
                (df['a0900i'] == 1)
            ),
            "No vive con pareja pero vive con otros adultos",
            np.where(
                df['a0900a'] != -99,
                "No vive con pareja ni con otros adultos",
                None   # millor que np.nan per evitar conflictes
            )
        )
    )
).astype(object)


In [6]:
df = df.drop(columns=[
    'a0900a','a0900b','a0900c','a0900d','a0900e',
    'a0900f','a0900g','a0900h','a0900i','a0900j','a0900k','a1500'
])


#### Asignación nombres nuevos columnas

In [7]:
df = df.rename(columns={
    'a0400': 'Año_naci.',
    'a04': 'Edad',
    'a01': 'Año_entrev.',
    'a02': 'Mes_entrev.'
})


#### Asignación al grupo por educación

In [8]:
df['edu_level'] = df['a1100'].replace({
    1: "Primaria", 2: "Primaria", 3: "Primaria",
    4: "Secundaria", 5: "Secundaria",
    6: "Universitaria", 7: "Universitaria", 8: "Universitaria", 9: "Universitaria"
})

In [9]:
df = df.drop(columns=[
    'a1400','a1600','a1700','a0300','a0800','a0100' 
])


### Indicamos CCAA

In [10]:
df['CCAA'] = df['a0200'].replace({
    1: "Andalusia", 2: "Aragon", 3: "Asturias",4: "Baleric Islands", 5: "Canary Island",6: "Cantabria", 7: "Castille-Leon", 8: "Castille-la Mancha", 9: "Catalonia",10:'Valencia Country',11:'Extremadura',12:'Galicia',13:'Madrid Region',14:'Murcia Region',15:'Navarre',16:'Basque Country',17:'La Rioja',-96:'(NS/NC)',-98:'(N/A)'
})

### Indicamos oficios

In [11]:
# dicionari 
ocupaciones_cno11 = {
    -99: "Missing / Error",
    -98: "No aplicable",
    -5: "No disponible",
    0: "Fuerzas Armadas",

    # Grupo 1 – Directivos y gerentes
    11: "Directores generales y gerentes",
    12: "Directores de departamento",
    13: "Gerentes de empresas pequeñas",
    14: "Gerentes comerciales",
    15: "Gerentes de hostelería y restauración",

    # Grupo 2 – Profesionales científicos e intelectuales
    21: "Profesionales de ciencias y ingeniería",
    22: "Profesionales de la salud",
    23: "Profesores y docentes",
    24: "Profesionales de economía, derecho y sociales",
    25: "Profesionales TIC",
    26: "Profesionales artísticos y culturales",
    27: "Profesionales del deporte",
    28: "Profesionales de información y comunicación",
    29: "Otros profesionales",

    # Grupo 3 – Técnicos y profesionales de apoyo
    31: "Técnicos de ciencias y ingeniería",
    32: "Técnicos de salud",
    33: "Técnicos de educación",
    34: "Técnicos de economía y administración",
    35: "Técnicos TIC",
    36: "Técnicos artísticos",
    37: "Técnicos del deporte",
    38: "Técnicos de información y comunicación",

    # Grupo 4 – Empleados administrativos
    41: "Administrativos generales",
    42: "Administrativos financieros",
    43: "Administrativos de apoyo",
    44: "Recepcionistas",
    45: "Otros administrativos",

    # Grupo 5 – Trabajadores de servicios y vendedores
    50: "Servicios diversos",
    51: "Personal de restauración",
    52: "Personal de venta",
    53: "Personal de cuidados y atención",
    54: "Personal de seguridad",
    55: "Personal de limpieza",
    56: "Servicios personales",
    57: "Servicios de protección",
    58: "Servicios de transporte",
    59: "Otros servicios",

    # Grupo 6 – Agricultura, ganadería, pesca
    61: "Agricultura",
    62: "Ganadería",
    63: "Silvicultura",
    64: "Pesca",

    # Grupo 7 – Artesanos y trabajadores cualificados
    71: "Construcción",
    72: "Metalurgia",
    73: "Electricidad y electrónica",
    74: "Mecánica",
    75: "Madera y muebles",
    76: "Textil y confección",
    77: "Alimentación",
    78: "Otros oficios",

    # Grupo 8 – Operadores de maquinaria
    81: "Operadores de instalaciones fijas",
    82: "Operadores de maquinaria móvil",
    83: "Conductores de vehículos",
    84: "Operadores de equipos industriales",

    # Grupo 9 – Ocupaciones elementales
    91: "Personal de limpieza",
    92: "Ayudantes de cocina y servicios",
    93: "Repartidores",
    94: "Peones de construcción",
    95: "Peones agrícolas",
    96: "Peones industriales",
    97: "Recolectores y manipuladores",
    98: "Ocupaciones elementales diversas"
}


In [12]:
#aplicar oficis
df["ocupacion"] = df["a1800"].replace(ocupaciones_cno11)


### Tu trabajo esta relacionado con aspectos financieros

In [13]:
df["trabajo_rel."]=df['a1900'].replace({0:"No",1:"Si",-96:"NS/NC",-98:"N/A"})

### hombre o mujer

In [14]:
df["gender"]=df['a0000'].replace({0:"Mujer",1:"Hombre",-96:"NS/NC",-98:"N/A"})

In [15]:
df=df.drop(columns=['a0200','a1800','a1900','a0000','a1100','a1200','a1300'])

In [16]:
df=df.drop(columns=['a05','a1000'])

In [17]:
#eliminar duplicados
df = df.loc[:, ~df.columns.duplicated()]

In [19]:
df.head()

,Año_entrev.,Mes_entrev.,Año_naci.,Edad,labor_status,familia,edu_level,CCAA,ocupacion,trabajo_rel.,gender
0,2016,12,1998,18,otros inactivos,No vive con pareja pero vive con otros adultos,Secundaria,Andalusia,No aplicable,N/A,Mujer
1,2016,10,1982,33,missing,Vive con pareja,Universitaria,Andalusia,Otros administrativos,Si,Mujer
2,2016,10,1967,49,otros inactivos,Vive con pareja,Primaria,Andalusia,Silvicultura,No,Hombre
3,2016,10,1962,54,otros inactivos,Vive con pareja,Primaria,Andalusia,Peones agrícolas,No,Mujer
4,2016,10,1952,64,jubilado,Vive con pareja,Primaria,Andalusia,Alimentación,No,Mujer


In [20]:
df.to_csv("C:/Users/Portátil/ProjecteData/Equip_23/Data/8 April/01_Demografia_cleaning_16.csv",index=False)